# Full-dataset Processed PPG — Rosenstein LLE

Production pipeline: load every segmented session, retain every Processed PPG window with `analysis_included=True`, compute one Rosenstein LLE result per window, write one audit-ready CSV per session, then merge all session files. Divergence arrays are intentionally not stored in the primary CSVs.

Output directory: `phase1/results/lle/processed/`.

In [ ]:
# Cell 1 — Setup, frozen configuration, and output schema
import sys
from importlib import import_module
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start):
    """Find the repository root from any notebook working directory."""
    for path in (start, *start.parents):
        if (path / "phase1" / "src").is_dir():
            return path
    raise FileNotFoundError("Repository root was not found.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
repo_path = str(REPO_ROOT)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

loader = import_module("phase1.src.dataloader.loader")
lyapunov = import_module("phase1.src.chaos.lyapunov")
load_segmented_session = loader.load_segmented_session
compute_rosenstein_lle = lyapunov.compute_rosenstein_lle
estimate_mean_period = lyapunov.estimate_mean_period

REPRESENTATION = "processed"
WINDOW_SIZES_S = (60, 120, 180)
STATE_NAMES = {0: "Awake", 1: "Drowsy"}
M = 8
TAU_S = 0.16
FIT_START_S = 0.80
FIT_END_S = 1.30
MAX_FOLLOW_S = 5.0
THEILER_RULE = "spectral_mean_period"
MIN_INITIAL_PAIRS = 50
MIN_FIT_PAIRS = 30
MIN_R2 = 0.90
OVERWRITE_EXISTING = True

SEGMENTED_DATA_DIR = REPO_ROOT / "phase1" / "segmentated_data" / "dhdata"
OUTPUT_DIR = REPO_ROOT / "phase1" / "results" / "lle" / "processed"
MERGED_FILENAME = "all_sessions_processed_rosenstein_lle.csv"

CSV_COLUMNS = [
    "session",
    "window_id",
    "state",
    "window_size_s",
    "representation",
    "sampling_rate_hz",
    "n_samples",
    "m",
    "tau_s",
    "tau_samples",
    "theiler_s",
    "theiler_samples",
    "fit_start_s",
    "fit_end_s",
    "fit_duration_s",
    "max_follow_s",
    "lle_1_per_s",
    "fit_r2",
    "n_embedded",
    "n_pairs_initial",
    "n_pairs_fit_min",
    "analysis_included",
    "signal_finite",
    "valid",
    "qc_reason",
]
IDENTITY_COLUMNS = [
    "session", "window_size_s", "window_id", "representation"
]

assert FIT_END_S <= MAX_FOLLOW_S
assert 0.0 <= FIT_START_S < FIT_END_S
assert SEGMENTED_DATA_DIR.is_dir()

print(f"Input: {SEGMENTED_DATA_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Schema: {len(CSV_COLUMNS)} columns")

In [ ]:
# Cell 2 — Discovery, validation, and per-session pipeline
def discover_session_ids(data_dir):
    """Return sorted numeric IDs for every exported session archive."""
    session_ids = []
    for path in data_dir.glob("sample_*.npz"):
        suffix = path.stem.removeprefix("sample_")
        if suffix.isdigit():
            session_ids.append(int(suffix))
    session_ids = sorted(set(session_ids))
    if not session_ids:
        raise FileNotFoundError(f"No session archives found in {data_dir}")
    return session_ids


def session_output_path(session_id):
    """Return the canonical CSV path for one session."""
    return OUTPUT_DIR / f"session_{session_id:02d}_processed_rosenstein_lle.csv"


def write_csv_atomic(frame, output_path):
    """Write a CSV atomically within its destination directory."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = output_path.with_suffix(output_path.suffix + ".tmp")
    frame.to_csv(temporary_path, index=False)
    temporary_path.replace(output_path)


def validate_lle_frame(frame, expected_rows=None, session_id=None):
    """Validate schema, identity, frozen configuration, and QC invariants."""
    if list(frame.columns) != CSV_COLUMNS:
        raise ValueError("LLE CSV schema or column order is invalid.")
    if expected_rows is not None and len(frame) != expected_rows:
        raise ValueError(
            f"Row-count mismatch: expected {expected_rows}, found {len(frame)}."
        )
    if frame.empty:
        raise ValueError("LLE output is empty.")
    if frame.duplicated(IDENTITY_COLUMNS).any():
        raise ValueError("Duplicate window identities detected.")
    if session_id is not None and not (frame["session"] == session_id).all():
        raise ValueError(f"Unexpected session value in Session {session_id} output.")
    if not (frame["representation"] == REPRESENTATION).all():
        raise ValueError("Output contains a non-processed representation.")
    if not frame["window_size_s"].isin(WINDOW_SIZES_S).all():
        raise ValueError("Output contains an unsupported window size.")
    if not frame["state"].isin(STATE_NAMES.values()).all():
        raise ValueError("Output contains an unsupported state.")
    if not frame["analysis_included"].astype(bool).all():
        raise ValueError("An excluded upstream window entered the output.")
    if not frame["signal_finite"].astype(bool).all():
        raise ValueError("A selected signal contains NaN or Inf.")
    if not (frame["m"] == M).all() or not np.allclose(frame["tau_s"], TAU_S):
        raise ValueError("Embedding configuration changed unexpectedly.")
    expected_tau = np.rint(frame["sampling_rate_hz"] * TAU_S).astype(int)
    if not np.array_equal(frame["tau_samples"].to_numpy(int), expected_tau):
        raise ValueError("tau_samples is inconsistent with tau_s and fs.")
    expected_theiler = np.maximum(
        np.rint(frame["theiler_s"] * frame["sampling_rate_hz"]).astype(int), 1
    )
    if not np.array_equal(frame["theiler_samples"].to_numpy(int), expected_theiler):
        raise ValueError("theiler_samples is inconsistent with mean-period theiler_s.")
    if not np.allclose(frame["fit_start_s"], FIT_START_S):
        raise ValueError("fit_start_s changed unexpectedly.")
    if not np.allclose(frame["fit_end_s"], FIT_END_S):
        raise ValueError("fit_end_s changed unexpectedly.")
    if not np.allclose(frame["fit_duration_s"], FIT_END_S - FIT_START_S):
        raise ValueError("fit_duration_s is inconsistent.")
    if not np.allclose(frame["max_follow_s"], MAX_FOLLOW_S):
        raise ValueError("max_follow_s changed unexpectedly.")
    expected_embedded = (
        frame["n_samples"] - (frame["m"] - 1) * frame["tau_samples"]
    )
    if not np.array_equal(frame["n_embedded"], expected_embedded):
        raise ValueError("n_embedded is inconsistent with the embedding.")
    valid_rows = frame["valid"].astype(bool)
    if not np.isfinite(
        frame.loc[valid_rows, ["lle_1_per_s", "fit_r2"]].to_numpy(float)
    ).all():
        raise ValueError("A valid row has a non-finite LLE or R².")
    if not (frame.loc[valid_rows, "fit_r2"] >= MIN_R2).all():
        raise ValueError("A valid row violates the minimum R² rule.")
    if not (frame.loc[valid_rows, "qc_reason"] == "ok").all():
        raise ValueError("A valid row has a non-ok QC reason.")


def run_lle(session_id, overwrite=OVERWRITE_EXISTING):
    """Compute, validate, and save all included Processed windows."""
    output_path = session_output_path(session_id)
    if output_path.exists() and not overwrite:
        cached = pd.read_csv(output_path)
        validate_lle_frame(cached, session_id=session_id)
        return cached, {"session": session_id, "status": "cached", "path": output_path}

    session_file = f"sample_{session_id}.csv"
    batches, metadata = load_segmented_session(
        session_file,
        data_dir=SEGMENTED_DATA_DIR,
        window_sizes=WINDOW_SIZES_S,
        representation=REPRESENTATION,
        stationarity_only=False,
    )
    if metadata.attrs["session"] != session_file:
        raise ValueError(f"Session metadata mismatch for {session_file}.")

    rows = []
    expected_rows = 0
    started = perf_counter()
    for window_size_s in WINDOW_SIZES_S:
        batch = batches[window_size_s]
        included = np.asarray(batch["stationarity_pass"], dtype=bool)
        expected_rows += int(np.sum(included))
        sampling_rate_hz = float(batch["fs"])
        tau_samples = max(int(np.round(TAU_S * sampling_rate_hz)), 1)

        for index in np.flatnonzero(included):
            signal = np.asarray(batch["signal"][index], dtype=float)
            signal_finite = bool(np.all(np.isfinite(signal)))
            if not signal_finite:
                raise ValueError(
                    f"Non-finite signal: session={session_id}, "
                    f"size={window_size_s}, window={batch['window_id'][index]}."
                )
            label_code = int(batch["label"][index])
            if label_code not in STATE_NAMES:
                raise ValueError(f"Unsupported label code: {label_code}")

            theiler_s = estimate_mean_period(signal, sampling_rate_hz)
            result = compute_rosenstein_lle(
                signal,
                sampling_rate=sampling_rate_hz,
                m=M,
                tau_samples=tau_samples,
                fit_start_s=FIT_START_S,
                fit_end_s=FIT_END_S,
                max_follow_s=MAX_FOLLOW_S,
                theiler_s=theiler_s,
                min_initial_pairs=MIN_INITIAL_PAIRS,
                min_fit_pairs=MIN_FIT_PAIRS,
                min_r2=MIN_R2,
            )
            rows.append({
                "session": session_id,
                "window_id": int(batch["window_id"][index]),
                "state": STATE_NAMES[label_code],
                "window_size_s": window_size_s,
                "representation": REPRESENTATION,
                "sampling_rate_hz": sampling_rate_hz,
                "n_samples": signal.size,
                "m": M,
                "tau_s": TAU_S,
                "tau_samples": tau_samples,
                "theiler_s": theiler_s,
                "theiler_samples": result.theiler_samples,
                "fit_start_s": FIT_START_S,
                "fit_end_s": FIT_END_S,
                "fit_duration_s": FIT_END_S - FIT_START_S,
                "max_follow_s": MAX_FOLLOW_S,
                "lle_1_per_s": result.lle,
                "fit_r2": result.fit_r2,
                "n_embedded": result.n_embedded,
                "n_pairs_initial": result.n_pairs_initial,
                "n_pairs_fit_min": result.n_pairs_fit_min,
                "analysis_included": True,
                "signal_finite": signal_finite,
                "valid": result.valid,
                "qc_reason": result.qc_reason,
            })

    frame = (
        pd.DataFrame(rows, columns=CSV_COLUMNS)
        .sort_values(["window_size_s", "window_id"])
        .reset_index(drop=True)
    )
    validate_lle_frame(frame, expected_rows=expected_rows, session_id=session_id)
    write_csv_atomic(frame, output_path)
    reloaded = pd.read_csv(output_path)
    validate_lle_frame(reloaded, expected_rows=expected_rows, session_id=session_id)
    elapsed_s = perf_counter() - started
    report = {
        "session": session_id,
        "n_windows": len(frame),
        "n_valid": int(frame["valid"].sum()),
        "elapsed_s": elapsed_s,
        "status": "written",
        "path": output_path,
    }
    return frame, report

In [ ]:
# Cell 3 — Discover the complete exported dataset
SESSIONS = discover_session_ids(SEGMENTED_DATA_DIR)
print(f"Discovered {len(SESSIONS)} sessions: {SESSIONS}")

index_table = pd.read_csv(SEGMENTED_DATA_DIR / "segments_index.csv")
index_table["analysis_included"] = index_table["stationarity_pass_processed"].astype(bool)
expected_index_rows = index_table[index_table["analysis_included"]].copy()
expected_session_counts = (
    expected_index_rows.assign(
        session_id=expected_index_rows["session"].str.extract(r"(\d+)").astype(int)
    )
    .groupby("session_id")
    .size()
)

assert set(SESSIONS) == set(expected_session_counts.index)
print(f"Expected included windows: {len(expected_index_rows)}")
display(
    expected_index_rows.groupby(["window_size_s", "label"])
    .size()
    .rename("n_windows")
    .to_frame()
)

In [ ]:
# Cell 4 — Run every session and write one CSV per session
session_frames = []
run_reports = []
full_run_started = perf_counter()

for position, session_id in enumerate(SESSIONS, start=1):
    print(f"[{position:02d}/{len(SESSIONS):02d}] Session {session_id:02d}: START", flush=True)
    frame, report = run_lle(session_id)
    expected_count = int(expected_session_counts.loc[session_id])
    if len(frame) != expected_count:
        raise ValueError(
            f"Session {session_id}: expected {expected_count}, found {len(frame)} rows."
        )
    session_frames.append(frame)
    run_reports.append(report)
    print(
        f"[{position:02d}/{len(SESSIONS):02d}] Session {session_id:02d}: "
        f"{report['status'].upper()} | rows={len(frame)} | "
        f"valid={int(frame['valid'].sum())} | elapsed={report.get('elapsed_s', 0.0):.1f}s",
        flush=True,
    )

all_results = (
    pd.concat(session_frames, ignore_index=True)
    .sort_values(["session", "window_size_s", "window_id"])
    .reset_index(drop=True)
)
validate_lle_frame(all_results, expected_rows=len(expected_index_rows))
merged_path = OUTPUT_DIR / MERGED_FILENAME
write_csv_atomic(all_results, merged_path)
merged_reloaded = pd.read_csv(merged_path)
validate_lle_frame(merged_reloaded, expected_rows=len(expected_index_rows))

run_report_table = pd.DataFrame(run_reports)
total_elapsed_s = perf_counter() - full_run_started
print(f"Completed {len(all_results)} windows in {total_elapsed_s:.1f} s.")
print(f"Merged CSV: {merged_path}")
display(run_report_table.drop(columns="path").round(3))

In [ ]:
# Cell 5 — Final schema, coverage, and QC audit
session_csvs = sorted(OUTPUT_DIR.glob("session_*_processed_rosenstein_lle.csv"))
assert len(session_csvs) == len(SESSIONS)
assert merged_path.is_file()
assert len(all_results) == len(expected_index_rows)
assert all_results[IDENTITY_COLUMNS].drop_duplicates().shape[0] == len(all_results)

coverage_table = (
    all_results.groupby(["window_size_s", "state"], as_index=False)
    .agg(
        n_windows=("window_id", "size"),
        n_valid=("valid", "sum"),
        median_lle_1_per_s=("lle_1_per_s", "median"),
        median_fit_r2=("fit_r2", "median"),
        min_pairs_fit=("n_pairs_fit_min", "min"),
    )
)
qc_table = (
    all_results.groupby(["valid", "qc_reason"], as_index=False)
    .size()
    .rename(columns={"size": "n_windows"})
)
sampling_table = (
    all_results.assign(
        sampling_rate_group_hz=all_results["sampling_rate_hz"].round(),
    )
    .groupby(["sampling_rate_group_hz", "tau_samples"], as_index=False)
    .size()
    .rename(columns={"size": "n_windows"})
)

print(f"Per-session CSVs: {len(session_csvs)}")
print(f"Merged rows: {len(all_results)}")
print(f"Valid rows: {int(all_results['valid'].sum())}")
display(coverage_table.round(5))
display(qc_table)
display(sampling_table)
display(all_results.head())

## Output contract

- One row is one upstream-included Processed PPG window.
- One file is written per discovered session as `session_XX_processed_rosenstein_lle.csv`.
- `all_sessions_processed_rosenstein_lle.csv` is the validated concatenation of all session files.
- `fit_start_s`, `fit_end_s`, and all estimator settings are repeated in every row for reproducibility.
- `time_s`, `mean_log_distance`, and `n_pairs_by_lag` are not stored in the primary CSVs.
- This pipeline produces window-level measurements and QC only; it performs no statistical inference.